# 02d — Weekly macro feature group (leak-corrected monthly lags)

Produces `data/processed/macro_weekly_lags.csv`: the four monthly macro
variables (`cpi`, `fed_funds`, `ind_prod`, `real_rates`) expressed as **3 lagged
weekly columns each** — 12 columns total — for the weekly model notebooks
(`04_random_forest`, `05_xgboost`, `06_lstm`) to consume as an optional `+Macro`
ablation rung.

**Why raw lags, not MIDAS.** MIDAS's Beta/Almon weight polynomial is a parsimony
device for *linear* models — it collapses several monthly lags into 2 shape
parameters. Tree models and LSTMs do not need it: given the raw lagged monthly
values they learn their own lag weighting (an RF on raw lags is essentially
U-MIDAS). MIDAS keeps its place as a standalone competing model in
`03_midas.ipynb`; here the other models simply get the raw lags.

**Look-ahead control.** `monthly_macro.csv` stamps each value on the 1st of its
reference month, but the figure is not published until weeks later. Each
variable is given a conservative publication lag; a value is only visible to a
weekly row once its `available_date` falls on or before that week's
position-taking Friday $F_{t-1}$. This mirrors the timing logic in
`03_midas.ipynb` §3 — see there for the full rationale and the data-revision
limitation.

In [ ]:
import pandas as pd
import numpy as np

macro = pd.read_csv('../data/raw/monthly_macro.csv', index_col=0, parse_dates=True)

MACRO_VARS   = [c for c in ['cpi', 'fed_funds', 'ind_prod', 'real_rates'] if c in macro.columns]
N_MACRO_LAGS = 3

# Conservative publication lag (days) added to each variable's month-1st stamp,
# turning a reference-month date into a real-time availability date.
# Mirrors MACRO_AVAIL_LAG in 03_midas.ipynb section 3.
MACRO_AVAIL_LAG = {'cpi': 46, 'ind_prod': 48, 'fed_funds': 35, 'real_rates': 46}

# Weekly W-FRI grid spanning the raw data.
weekly_index = pd.date_range(macro.index.min(), macro.index.max(), freq='W-FRI')


def build_weekly_macro_lags(weekly_dates, series, lag_days, n_lags=N_MACRO_LAGS):
    """For each weekly Friday, the n_lags most recent monthly observations whose
    availability date (stamp + lag_days) falls on or before the prior Friday
    F_{t-1} = week_end - 7 -- so macro is observed at position-taking time."""
    s     = series.dropna().sort_index()
    avail = s.index + pd.Timedelta(days=lag_days)
    out   = np.full((len(weekly_dates), n_lags), np.nan)
    for i, d in enumerate(weekly_dates):
        past = s.values[avail < (d - pd.Timedelta(days=6))]   # observable by F_{t-1}
        if len(past) >= n_lags:
            out[i] = past[-n_lags:][::-1]                     # mlag1 = most recent
    return out


frames = []
for v in MACRO_VARS:
    mat  = build_weekly_macro_lags(weekly_index, macro[v], MACRO_AVAIL_LAG[v])
    cols = [f'{v}_mlag{k}' for k in range(1, N_MACRO_LAGS + 1)]
    frames.append(pd.DataFrame(mat, index=weekly_index, columns=cols))

macro_weekly = pd.concat(frames, axis=1)
macro_weekly.index.name = 'week_end'
macro_weekly.to_csv('../data/processed/macro_weekly_lags.csv')

print(f'macro_weekly_lags.csv  ->  {macro_weekly.shape[0]} weeks x {macro_weekly.shape[1]} columns')
print('Publication lags (days):', MACRO_AVAIL_LAG)
print('Columns:', list(macro_weekly.columns))
macro_weekly.dropna().tail(4)

## Look-ahead audit

Independently re-derives the monthly observation behind every `*_mlag1` and
asserts its availability date never postdates the position Friday $F_{t-1}$.
The spot check then shows, for the most recent weeks, which reference month
feeds `cpi_mlag1` and when it became observable — so the guarantee is visible,
not just claimed.

In [ ]:
position_fri = weekly_index - pd.Timedelta(days=7)

for v in MACRO_VARS:
    s     = macro[v].dropna().sort_index()
    avail = s.index + pd.Timedelta(days=MACRO_AVAIL_LAG[v])
    min_slack = np.inf
    for i, d in enumerate(weekly_index):
        used = avail[avail < (d - pd.Timedelta(days=6))]
        if len(used):
            min_slack = min(min_slack, (position_fri[i] - used.max()).days)
    assert (not np.isfinite(min_slack)) or min_slack >= 0, f'Look-ahead in {v} (slack {min_slack}d)'
    slack_str = f'{int(min_slack)}d' if np.isfinite(min_slack) else 'n/a'
    print(f'  {v:<11} OK  (tightest week: value available {slack_str} before the position Friday)')

print('\nSpot check - cpi_mlag1 source month vs forecast week:')
cpi       = macro['cpi'].dropna().sort_index()
cpi_avail = cpi.index + pd.Timedelta(days=MACRO_AVAIL_LAG['cpi'])
for d in macro_weekly['cpi_mlag1'].dropna().index[-4:]:
    src = cpi.index[cpi_avail < (d - pd.Timedelta(days=6))].max()
    print(f'  forecast week {d.date()}  ->  cpi from {src.strftime("%Y-%m")}'
          f'  (available ~{(src + pd.Timedelta(days=MACRO_AVAIL_LAG["cpi"])).date()},'
          f' position Fri {(d - pd.Timedelta(days=7)).date()})')

print('\nLook-ahead audit passed.')